# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hussaintinwala2/Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### My method choice

I will use **Logistic Regression** for the modeling task.

The goal is to prioritize content pages for review or refresh using signals available at the decision point. Logistic Regression fits this lane because it provides an interpretable probability for the target outcome and allows the contribution of each feature to be inspected.

I prefer this method as a first model because it is relatively simple and less prone to unnecessary complexity. It also gives a useful comparison against my Week-4 rule-based baseline: if a learned model cannot improve on the simple baseline, adding a more complex model may not be justified.

The model will use only information available at the decision point. No future-window outcomes or baseline/product flags will be used as input features.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Method choice check

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print("Selected method: Logistic Regression")
print("Reason: interpretable, probability-based, and suitable for a first learned model.")
print("Leakage rule: only decision-point features will be used.")

Selected method: Logistic Regression
Reason: interpretable, probability-based, and suitable for a first learned model.
Leakage rule: only decision-point features will be used.


In [29]:
# Reconnect to the FlyRank warehouse

import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [30]:
# Prepare the modeling dataset
import numpy as np
# Pull the required decision-point fields from the warehouse
model_data = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        c.content_created_date,
        c.content_updated_date
    FROM {TABLES['fact_daily']} f
    LEFT JOIN {TABLES['dim_content']} c
        ON f.client_hash_id = c.client_hash_id
        AND f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= DATE '2026-03-01'
      AND f.report_date < DATE '2026-04-01'
""").df()

# Calculate decision-point features
model_data["days_since_update"] = (
    model_data["report_date"] -
    model_data["content_updated_date"]
).dt.days

model_data["content_age_days"] = (
    model_data["report_date"] -
    model_data["content_created_date"]
).dt.days

# Calculate CTR from observed GSC data
model_data["ctr"] = (
    model_data["gsc_clicks"] /
    model_data["gsc_impressions"].replace(0, np.nan)
)

# Keep only rows with usable feature values
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "days_since_update",
    "content_age_days",
    "ctr"
]

model_data = model_data.dropna(
    subset=feature_cols + ["client_hash_id"]
).copy()

print(f"Modeling rows: {len(model_data):,}")
print(f"Clients: {model_data['client_hash_id'].nunique():,}")
print("\nFeatures:")
print(feature_cols)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 3,611,061
Clients: 47

Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'days_since_update', 'content_age_days', 'ctr']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### My split design

I will use a **grouped train/test split by `client_hash_id`**.

The modeling dataset contains one row per client × content page. March 2026 search-performance data is used as the decision-point input, while April 2026 performance is used only to create the future outcome label.

I will keep 75% of clients for training and 25% for testing, using a fixed random seed for reproducibility. No client will appear in both sets.

This split tests whether the model can generalize to content pages from clients it did not see during training, rather than benefiting from having pages from the same client in both training and test data.


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

# Use a fixed seed so the split is reproducible
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_data,
        groups=model_data["client_hash_id"]
    )
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()

train_clients = set(train_data["client_hash_id"])
test_clients = set(test_data["client_hash_id"])

print(f"Training rows: {len(train_data):,}")
print(f"Test rows:     {len(test_data):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Test clients:     {len(test_clients):,}")
print(f"Client overlap:   {len(train_clients & test_clients)}")

assert len(train_clients & test_clients) == 0

print("\nVerified: no client appears in both train and test sets.")

Training rows: 2,625,169
Test rows:     985,892
Training clients: 35
Test clients:     12
Client overlap:   0

Verified: no client appears in both train and test sets.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model target and comparison

The model will predict whether a content page meets the defined review-priority outcome.

The Logistic Regression model will be trained only on March decision-point features. The Week-4 baseline is recreated using the same March test pages and its original scoring rule. April performance is used only to construct the evaluation label The Week-4 rule will be evaluated on the same test set as the model.

I will compare the two approaches using **ROC-AUC**, which measures how well the ranking separates higher-priority from lower-priority pages across different score thresholds.

The comparison is intended as decision-support evidence rather than proof that the model will improve future performance.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the available date range in the warehouse

date_range = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM {TABLES['fact_daily']}
""").df()

print(date_range)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  first_date  last_date  distinct_dates
0 2025-01-27 2026-06-30             520


In [33]:
# Build one row per client × content page for the March decision point.
# Features come from March 2026.
# Outcome comes from April 2026.

march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        AVG(gsc_avg_position) AS avg_position_march

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY client_hash_id, content_hash_id
""").df()

april_outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april,
        SUM(gsc_clicks) AS clicks_april

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-04-01'
      AND report_date < DATE '2026-05-01'

    GROUP BY client_hash_id, content_hash_id
""").df()

model_page = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print(f"March content pages with April outcomes: {len(model_page):,}")
print(f"Clients represented: {model_page['client_hash_id'].nunique():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March content pages with April outcomes: 331,436
Clients represented: 55


In [34]:
# Recreate the future outcome label

model_page["is_declining"] = (
    model_page["impressions_april"]
    < 0.80 * model_page["impressions_march"]
).astype(int)

print("Target created successfully.")
print(model_page["is_declining"].value_counts())
print(f"Positive rate: {model_page['is_declining'].mean():.3f}")

Target created successfully.
is_declining
0    237437
1     93999
Name: count, dtype: int64
Positive rate: 0.284


In [35]:
from sklearn.model_selection import GroupShuffleSplit

# Features available at the March decision point
feature_cols = [
    "impressions_march",
    "clicks_march",
    "avg_position_march"
]

# Remove rows with missing model inputs
model_page = model_page.dropna(
    subset=feature_cols + ["is_declining", "client_hash_id"]
).copy()

# Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_page,
        y=model_page["is_declining"],
        groups=model_page["client_hash_id"]
    )
)

train_page = model_page.iloc[train_idx].copy()
test_page = model_page.iloc[test_idx].copy()

train_clients = set(train_page["client_hash_id"])
test_clients = set(test_page["client_hash_id"])

print(f"Training pages: {len(train_page):,}")
print(f"Test pages:     {len(test_page):,}")
print(f"Training clients: {len(train_clients)}")
print(f"Test clients:     {len(test_clients)}")
print(f"Client overlap:   {len(train_clients & test_clients)}")

assert len(train_clients & test_clients) == 0

print("\nVerified: train and test clients do not overlap.")

Training pages: 133,473
Test pages:     43,264
Training clients: 35
Test clients:     12
Client overlap:   0

Verified: train and test clients do not overlap.


In [36]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train = train_page[feature_cols]
y_train = train_page["is_declining"]

X_test = test_page[feature_cols]
y_test = test_page["is_declining"]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]

model_auc = roc_auc_score(y_test, model_prob)

print(f"Logistic Regression ROC-AUC: {model_auc:.3f}")

Logistic Regression ROC-AUC: 0.538


In [37]:
# Recreate the exact Week-4 baseline score on the March test pages.
# This uses only March decision-point information.

baseline_test = test_page[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_march",
        "clicks_march"
    ]
].copy()

# Calculate March CTR using the same definition as Week 4.
baseline_test["ctr"] = (
    baseline_test["clicks_march"] /
    baseline_test["impressions_march"].replace(0, np.nan)
)

# Get content update dates for the same pages.
content_dates = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date
    FROM {TABLES['dim_content']}
""").df()

content_dates["content_updated_date"] = pd.to_datetime(
    content_dates["content_updated_date"],
    errors="coerce"
)

# Join update dates to the March test pages.
baseline_test = baseline_test.merge(
    content_dates,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Week-4 used 2026-03-31 as the decision date.
decision_date = pd.Timestamp("2026-03-31")

baseline_test["days_since_update"] = (
    decision_date -
    baseline_test["content_updated_date"]
).dt.days

# Recreate the exact Week-4 staleness score: 0–50.
baseline_test["staleness_score"] = (
    baseline_test["days_since_update"]
    .clip(lower=0, upper=365)
    / 365
    * 50
)

# Recreate the exact Week-4 CTR score: 0–50.
baseline_test["ctr_score"] = (
    ((0.05 - baseline_test["ctr"]) / 0.05)
    .clip(lower=0, upper=1)
    * 50
)

# Exact Week-4 priority score.
baseline_test["priority_score"] = (
    baseline_test["staleness_score"] +
    baseline_test["ctr_score"]
)

# Remove rows where the baseline cannot be calculated.
baseline_test = baseline_test.dropna(
    subset=["priority_score"]
).copy()

# Match the baseline rows to the same y_test rows.
baseline_eval = test_page[
    ["client_hash_id", "content_hash_id", "is_declining"]
].merge(
    baseline_test[
        ["client_hash_id", "content_hash_id", "priority_score"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

baseline_prob = baseline_eval["priority_score"].to_numpy()
baseline_y = baseline_eval["is_declining"].to_numpy()

baseline_auc = roc_auc_score(
    baseline_y,
    baseline_prob
)

print(f"Baseline evaluation rows: {len(baseline_eval):,}")
print(f"Logistic Regression test rows: {len(y_test):,}")
print(f"Week-4 baseline ROC-AUC: {baseline_auc:.3f}")
print(f"Logistic Regression ROC-AUC: {model_auc:.3f}")

Baseline evaluation rows: 43,264
Logistic Regression test rows: 43,264
Week-4 baseline ROC-AUC: 0.584
Logistic Regression ROC-AUC: 0.538


In [38]:
# Confirm that the baseline and model are being evaluated on the same pages.

model_keys = set(
    zip(
        test_page["client_hash_id"],
        test_page["content_hash_id"]
    )
)

baseline_keys = set(
    zip(
        baseline_eval["client_hash_id"],
        baseline_eval["content_hash_id"]
    )
)

print("Same evaluation pages:", model_keys == baseline_keys)

assert model_keys == baseline_keys
assert len(baseline_y) == len(y_test)

print("Verified: baseline and Logistic Regression use the same test pages.")

Same evaluation pages: True
Verified: baseline and Logistic Regression use the same test pages.


In [39]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 rule-based baseline",
        "Logistic Regression"
    ],
    "roc_auc": [
        baseline_auc,
        model_auc
    ]
})

comparison["roc_auc"] = comparison["roc_auc"].round(3)

display(comparison)

,method,roc_auc
0,Week-4 rule-based baseline,0.584
1,Logistic Regression,0.538


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The Logistic Regression model achieved a ROC-AUC of 0.538 on the client-grouped test set. This is only modestly above the 0.500 random level, so the model provides weak predictive separation and should be treated as directional decision-support rather than a reliable predictor.

The model's strongest coefficient was `clicks_march` (-0.526), followed by `impressions_march` (0.143) and `avg_position_march` (-0.135). Because the inputs were standardized before Logistic Regression, the coefficient magnitudes can be compared as directional signals within this model.

The negative coefficient for clicks means that, within this model, higher March clicks were associated with a lower predicted probability of decline. Impressions had a smaller positive coefficient, while average position had a smaller negative coefficient.

The model made 22,506 incorrect predictions out of 43,264 test pages, giving an error rate of approximately 52%. The inspected errors include pages with substantial impressions but few clicks, as well as pages with relatively strong average positions. This shows that these three search-performance signals alone do not cleanly separate declining and non-declining pages.

Overall, the learned model does not provide a strong improvement in predictive separation. The result suggests that additional decision-point signals would be needed to build a more useful model, and that model complexity should not be increased without evidence that it improves the same held-out evaluation.

In [40]:
print(model)
print("\nPipeline steps:")
print(model.named_steps)

Pipeline(steps=[('scaler', StandardScaler()),
                ('logistic',
                 LogisticRegression(max_iter=1000, random_state=42))])

Pipeline steps:
{'scaler': StandardScaler(), 'logistic': LogisticRegression(max_iter=1000, random_state=42)}


In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect Logistic Regression coefficients
import pandas as pd
import numpy as np
# Inspect Logistic Regression coefficients inside the pipeline

logreg = model.named_steps["logistic"]

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logreg.coef_[0]
})

coef_table["abs_coefficient"] = coef_table["coefficient"].abs()

coef_table = coef_table.sort_values(
    "abs_coefficient",
    ascending=False
)

print("Feature coefficients:")
display(coef_table[["feature", "coefficient"]])

Feature coefficients:


,feature,coefficient
1,clicks_march,-0.525518
0,impressions_march,0.143301
2,avg_position_march,-0.135124


In [42]:
# Inspect Logistic Regression coefficients

# Inspect prediction errors

test_results = X_test.copy()

test_results["actual"] = y_test.values

test_results["predicted_prob"] = model.predict_proba(X_test)[:, 1]

test_results["predicted"] = (
    test_results["predicted_prob"] >= 0.5
).astype(int)

test_results["error"] = (
    test_results["actual"] != test_results["predicted"]
)

print("Test rows:", len(test_results))
print("Incorrect predictions:", test_results["error"].sum())
print(
    "Error rate:",
    round(test_results["error"].mean(), 3)
)

print("\nSample of incorrect predictions:")
display(
    test_results[test_results["error"]].head(10)
)

Test rows: 43264
Incorrect predictions: 22506
Error rate: 0.52

Sample of incorrect predictions:


,impressions_march,clicks_march,avg_position_march,actual,predicted_prob,predicted,error
0,1140.0,2.0,4.394234,0,0.586471,1,True
1,57.0,0.0,2.714744,0,0.592291,1,True
3,1421.0,6.0,6.320337,0,0.565009,1,True
4,2770.0,16.0,4.459107,0,0.527672,1,True
5,48.0,0.0,14.753175,0,0.569609,1,True
8,6048.0,23.0,4.950311,0,0.513597,1,True
13,4788.0,10.0,9.306264,0,0.561856,1,True
15,178.0,0.0,9.455300,0,0.580461,1,True
16,194.0,0.0,37.707873,0,0.526717,1,True
19,228.0,0.0,44.434505,0,0.513984,1,True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.